# Qwen3.5-4B pipeline SMOKE TEST — Colab (T4 x1)

**Colab T4 x1 (16 GB) cannot train 4B at 32K context.** This notebook is a
smoke test only: it runs the full pipeline (mix → load → loss patch → LoRA
→ 100 records at 2K context) to verify everything works, then you move to
**Kaggle T4 x2** for real training.

- `MAX_SEQ_LEN = 2048` and `LIMIT = 100` keep this inside 16 GB.
- The same fp16 / loss-patch / max_grad_norm settings as the Kaggle notebook.
- A passing smoke test means the Kaggle run will start cleanly.

Real training notebook: `qwen35_0.8b_kaggle.ipynb` (4B, 32K context, T4 x2).


In [ ]:
# Cell 1: Installation & Fixes
# This cell uses uv to install a known-good set of versions for Qwen3.5 on Colab.
import os, importlib.util

# Install uv first, as it's better at resolving complex dependencies
if importlib.util.find_spec("uv") is None:
    !pip install -q uv

# Install the core, compatible dependency stack in one command
!uv pip install --upgrade --no-deps "torch==2.8.0" "triton>=3.3.0" torchvision bitsandbytes xformers==0.0.32.post2 \
    "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo" \
    "unsloth[base] @ git+https://github.com/unslothai/unsloth"

# Explicitly pin the compatible tokenizers and transformers versions
!uv pip install --upgrade --no-deps "tokenizers>=0.22.0,<=0.23.0" trl==0.22.2 unsloth unsloth_zoo
!uv pip install "transformers==5.3.0"

# This is required for the flash-linear-attention kernels used by Qwen3.5
!uv pip install --no-build-isolation flash-linear-attention causal_conv1d==1.6.0

# This is needed for the Triton kernels
!uv pip install --no-deps --upgrade "torchao>=0.16.0"
print("Installation complete. Please RESTART the runtime before proceeding.")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.6/20.6 MB 69.3 MB/s eta 0:00:00
Using Python 3.13.15 environment at: /usr
⠙ unsloth-zoo==2026.9.9                                                         

In [4]:
# Cell 1.5: Fix Torchvision Compatibility Issue
# Unsloth recommends force-reinstalling torchvision==0.23.0 to align correctly with torch 2.8.0 on Colab
!pip install --force-reinstall --no-deps --no-cache-dir "torchvision==0.23.0"
print("Reinstallation complete. Please RESTART the runtime (Runtime -> Restart session) before executing imports.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.6/8.6 MB 63.4 MB/s eta 0:00:00
  Attempting uninstall: torchvision
    Found existing installation: torchvision 0.23.0
    Uninstalling torchvision-0.23.0:
      Successfully uninstalled torchvision-0.23.0
Reinstallation complete. Please RESTART the runtime (Runtime -> Restart session) before executing imports.


In [1]:
from unsloth import FastLanguageModel
import torch
from datasets import load_dataset
from trl import SFTTrainer
from transformers import TrainingArguments

# ── CONFIG ────────────────────────────────────────
DATASETS = [
    "kernel-syzfix-qwen35",
    "security-qa-qwen35",
]

BASE_MODEL = "unsloth/Qwen3.5-4B"
EXPERT_NAME = "code-review-lora"
MAX_SEQ_LEN = 4096
LIMIT = 200          # 0 = exhaust every mixed record
PUSH_TO_HUB = False
OUT_DIR = "/content"
SEED = 3407

# Colab T4 x1 — SMOKE TEST: 2K context, 100 records, batch 1, grad accum 4.
print("GPUs:", torch.cuda.device_count())
if torch.cuda.is_available():
    print("device:", torch.cuda.get_device_name(0))
    print("bf16 supported (will be False on T4):",
          torch.cuda.is_bf16_supported())

/usr/local/lib/python3.13/dist-packages/unsloth/_gpu_init.py:110: UserWarning: Unsloth: torchaudio cannot initialise against this torch and has been disabled for this process, so anything that needs it will report it as missing rather than crash at import. Install the matching wheel to restore it. Original error: /usr/local/lib/python3.13/dist-packages/torchaudio/lib/_torchaudio.abi3.so: undefined symbol: torch_library_impl
  disable_torchaudio_if_cuda_mismatched()


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


/usr/local/lib/python3.13/dist-packages/unsloth/_gpu_init.py:446: UserWarning: torchcodec 0.11.0+cu130 is incompatible with torch 2.8.0+cu128; install a matching build with `pip install --index-url https://download.pytorch.org/whl/cu128 'torchcodec>=0.7,<0.8.0'`.
  disable_torchcodec_if_broken()
/usr/local/lib/python3.13/dist-packages/unsloth/_gpu_init.py:446: UserWarning: Unsloth: torchcodec is installed but cannot load its native libraries although FFmpeg is on the loader path; likely an FFmpeg major it does not support (it takes 4 to 8), a missing CUDA NPP runtime (nvidia-npp), or a build that does not match this torch; audio datasets decode through soundfile and PyAV instead (wav/flac/mp3/ogg, m4a/aac/webm).
  disable_torchcodec_if_broken()


🦥 Unsloth Zoo will now patch everything to make training faster!
GPUs: 1
device: Tesla T4
bf16 supported (will be False on T4): False


In [3]:
# Cell: Clone the MoE-orchestrator repo so we can import the dataset mixer
import os, subprocess, sys

REPO_URL = "https://github.com/michaelowusuntim6/MoE-orchestrator.git"
REPO_DIR = "/content/MoE-orchestrator"

if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "-q", REPO_URL, REPO_DIR], check=True)

# Add the repo root to sys.path so `orchestrator` is importable
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

# Verify
import importlib
importlib.invalidate_caches()
from orchestrator.dataset_mixer import smart_mix, smart_mix_report
print("dataset_mixer imported OK")

dataset_mixer imported OK


In [4]:
from datasets import load_dataset
import sys, os

# Ensure the repo root is importable when running from Colab
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())

from orchestrator.dataset_mixer import smart_mix, smart_mix_report

# DATASETS is configured in the config cell:
#   str    -> single dataset, no mixing
#   list   -> exhaustively mix every record from every dataset
#
# The mixer sorts by size, uses whole-number ratios interleaved,
# then appends the truncated decimals in descending order so no
# record is lost. Deterministic given SEED.

if isinstance(DATASETS, str):
    ds = load_dataset(f"michaelowusuntim6/{DATASETS}", split="train")
    print(f"Single dataset: {len(ds)} records")
else:
    plan = smart_mix_report(DATASETS)
    print("Mix plan:")
    print(f"  sorted:   {plan['sorted']}")
    print(f"  smallest: {plan['smallest']}")
    print(f"  ratios:   {plan['ratios']}")
    print(f"  wholes:   {plan['wholes']}")
    print(f"  extras:   {plan['extras']}")
    print(f"  whole mix: {plan['whole_total']} records (interleaved)")
    print(f"  extras:    {plan['extras_total']} records (appended)")
    print(f"  final:     {plan['final_total']} records")
    ds = smart_mix(DATASETS, seed=SEED)
    print(f"Loaded {len(ds)} records")

if LIMIT and LIMIT < len(ds):
    ds = ds.select(range(LIMIT))
print(f"{len(ds)} training records after limit")

# Sanity: content must be a string (FastLanguageModel needs str)
sample = ds[0]["messages"][0]
print(f"First record content type: {type(sample['content']).__name__}")
assert isinstance(sample["content"], str), "content must be str, not list"


Mix plan:
  sorted:   [('debug-qwen35', 408372), ('code-review-qwen35', 228452)]
  smallest: 228452
  ratios:   [1.787561500884212, 1.0]
  wholes:   [1, 1]
  extras:   [0.787561500884212, 0.0]
  whole mix: 456904 records (interleaved)
  extras:    179920 records (appended)
  final:     636824 records


README.md:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

train.jsonl: reconstructing file:   0%|          |  0.00B /  801MB            

train.jsonl: downloading bytes:           |  0.00B            

val.jsonl: reconstructing file:   0%|          |  0.00B / 16.7MB            

...

val.jsonl: reconstructing file:   0%|          |  0.00B / 30.8MB            

val.jsonl: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/408372 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/8176 [00:00<?, ? examples/s]

dataset                                size   ratio  whole   extra
debug-qwen35                         408372    1.79      1    0.79
code-review-qwen35                   228452    1.00      1    0.00
------------------------------------------------------------------
whole mix: 456904 records (interleaved 1:1)
extras:    179920 records (concatenated in descending order)
final:     636824 records
smart_mix -> 636824 records (whole 456904 + extras 179920)
Loaded 636824 records
636824 training records after limit
First record content type: str


In [5]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LEN,
    dtype=None,             # auto -> fp16 on T4
    load_in_4bit=False,     # bf16/fp16 LoRA, not QLoRA
    device_map="auto",      # uses both T4s
)

# CRITICAL: verify Unsloth's chunked loss is patched (issue #5441).
# Without this, the stock ForCausalLMLoss does logits.float() and
# allocates ~30 GB at 32K seq_len -> OOM.
assert "Unsloth" in str(model.loss_function), \
    f"loss not patched (issue #5441): {model.loss_function}\n" \
    "Upgrade unsloth and unsloth_zoo, then re-run."
print("loss patch verified:", type(model.loss_function).__name__)


==((====))==  Unsloth 2026.9.14: Fast Qwen3_5 patching. Transformers: 5.3.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.4.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.32.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for qwen3_5 won't work! Using float32.
Unsloth: QLoRA and full finetuning all not selected. Switching to 16bit LoRA.


Loading weights:   0%|          | 0/723 [00:00<?, ?it/s]

loss patch verified: function


In [6]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    lora_dropout=0.0,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=SEED,
)


Unsloth: Explicit target_modules are constrained by the finetune_(vision|language|attention|mlp) filters; adapters attach only where both select.


In [7]:
def format_record(ex):
    ex["text"] = tokenizer.apply_chat_template(
        ex["messages"], tokenize=False, add_generation_prompt=False)
    return ex

ds = ds.map(format_record, num_proc=2)
print(len(ds), "formatted records")


Map (num_proc=2):   0%|          | 0/636824 [00:00<?, ? examples/s]

636824 formatted records


In [8]:
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=ds,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LEN,
    dataset_num_proc=2,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=1,       # 1 — do not raise at 32K
        gradient_accumulation_steps=4,       # smooths gradients
        num_train_epochs=3,
        learning_rate=1.5e-5,                  # 1.5e-5 for 4B; 2e-4 too hot
        bf16=False,                          # T4: no bf16
        fp16=True,                           # T4: fp16
        max_grad_norm=0.3,                   # tighter than default 1.0
        logging_steps=10,
        optim="paged_adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        warmup_ratio=0.1,
        output_dir=f"{OUT_DIR}/outputs/{EXPERT_NAME}",
        save_strategy="epoch",
        save_total_limit=1,
        seed=SEED,
        report_to="none",
    ),
)
trainer.train()


warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Unsloth: Switching to float32 training since model cannot work with float16


warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/636824 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 248046}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 636,824 | Num Epochs = 3 | Total steps = 477,618
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 4 x 1) = 4
 "-____-"     Trainable parameters = 21,233,664 of 4,560,499,200 (0.47% trained)


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss
10,1.763367
20,1.803466
30,1.988853
40,1.625031
50,1.771636
60,1.594926
70,1.628830
80,1.689357
90,1.848922
100,1.800450


KeyboardInterrupt: 

In [ ]:
model.save_pretrained(f"{OUT_DIR}/adapters/{EXPERT_NAME}")
tokenizer.save_pretrained(f"{OUT_DIR}/adapters/{EXPERT_NAME}")
print("Adapter saved to", f"{OUT_DIR}/adapters/{EXPERT_NAME}")
print(f"Adapter name: {EXPERT_NAME} (push to HF as michaelowusuntim6/{EXPERT_NAME})")


In [ ]:
if PUSH_TO_HUB:
    from huggingface_hub import login
    import os
    login(token=os.environ["HF_TOKEN"])
    model.push_to_hub(f"michaelowusuntim6/{EXPERT_NAME}")
    tokenizer.push_to_hub(f"michaelowusuntim6/{EXPERT_NAME}")
    print(f"Pushed: https://huggingface.co/michaelowusuntim6/{EXPERT_NAME}")
else:
    print("PUSH_TO_HUB is False — set to True to upload")


In [ ]:
FastLanguageModel.for_inference(model)
messages = [{"role": "user", "content": "How do I fix a memory leak in C?"}]
inputs = tokenizer.apply_chat_template(
    messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
).to("cuda")
out = model.generate(inputs, max_new_tokens=256, do_sample=False)
print(tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True))
